# LoRA Scan Sweep on Google Colab (Persistent)

**Features:**
- Auto-saves results to Google Drive after each run
- Early stopping to reduce training time
- Survives runtime disconnections
- Can resume from where it left off

**Setup:**
1. Runtime → Change runtime type → Select GPU (T4, L4, or A100)
2. Run all cells in order

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
import os

drive.mount('/content/drive')

# Create directory for results
DRIVE_DIR = '/content/drive/MyDrive/lora_scan_results'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Results will be saved to: {DRIVE_DIR}")

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone or update the repository
import os

if os.path.exists('lora_scan'):
    print("Repository exists, pulling latest changes...")
    !cd lora_scan && git pull
else:
    print("Cloning repository...")
    !git clone https://github.com/ubitquitin/lora_scan.git

%cd lora_scan

In [ ]:
# Install micromamba (lightweight conda alternative)
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!./bin/micromamba shell init -s bash -p ~/micromamba

In [ ]:
# Create environment and install dependencies
!./bin/micromamba create -n spar python=3.12 -y -c conda-forge
!./bin/micromamba run -n spar pip install -e .

In [ ]:
# Restore previous results from Google Drive if they exist
import os
import shutil

DRIVE_RUNS = f"{DRIVE_DIR}/runs"
LOCAL_RUNS = "/content/lora_scan/runs"

if os.path.exists(DRIVE_RUNS):
    print("Restoring previous results from Google Drive...")
    if os.path.exists(LOCAL_RUNS):
        shutil.rmtree(LOCAL_RUNS)
    shutil.copytree(DRIVE_RUNS, LOCAL_RUNS)
    print(f"Restored results. Resuming from previous progress.")
else:
    print("No previous results found. Starting fresh.")

In [ ]:
# Test the setup with a dry run
!./bin/micromamba run -n spar python scripts/run.py configs/experiments/sft_sweep_mmlu_gsm8k.yaml --dry-run | head -20

## Run the Sweep with Auto-Save

This cell will:
- Run each experiment in the sweep
- **Only save successful runs** to Google Drive
- Show progress: X/72 completed, Y failed
- Skip already-completed runs
- Survive disconnections (just re-run this cell to resume)

In [ ]:
%%bash
# Run sweep with smart auto-save (only saves successful runs)
DRIVE_DIR="/content/drive/MyDrive/lora_scan_results"
LOCAL_RUNS="/content/lora_scan/runs"

# Progress tracking
COMPLETED=0
FAILED=0
CURRENT_RUN=""

echo "=========================================="
echo "  Starting Sweep with Auto-Save"
echo "=========================================="
echo ""

# Run the sweep
./bin/micromamba run -n spar python scripts/run.py configs/experiments/sft_sweep_mmlu_gsm8k.yaml 2>&1 | while IFS= read -r line; do
    echo "$line"
    
    # Detect when a new run starts
    if echo "$line" | grep -q "\[.*TODO.*\]"; then
        CURRENT_RUN=$(echo "$line" | grep -oP '\[\d+/\d+\].*' || echo "")
    fi
    
    # Detect successful completion (when results are saved)
    if echo "$line" | grep -q "Training completed in"; then
        # Wait a moment for files to be written
        sleep 2
        
        # Check if the run actually succeeded by looking at metadata
        LAST_RUN_DIR=$(ls -td "$LOCAL_RUNS"/*/ 2>/dev/null | head -1)
        if [ -n "$LAST_RUN_DIR" ] && [ -f "${LAST_RUN_DIR}metadata.json" ]; then
            STATUS=$(grep -o '"status": *"[^"]*"' "${LAST_RUN_DIR}metadata.json" | cut -d'"' -f4)
            
            if [ "$STATUS" = "complete" ]; then
                COMPLETED=$((COMPLETED + 1))
                echo ""
                echo "✓ SUCCESS - Syncing to Google Drive..."
                rsync -a --delete "$LOCAL_RUNS/" "$DRIVE_DIR/runs/"
                echo "✓ Saved to Drive | Progress: $COMPLETED completed, $FAILED failed"
                echo ""
            elif [ "$STATUS" = "failed" ]; then
                FAILED=$((FAILED + 1))
                echo ""
                echo "✗ FAILED - NOT saving to Drive"
                echo "✗ Progress: $COMPLETED completed, $FAILED failed"
                echo ""
            fi
        fi
    fi
done

# Final sync and summary
echo ""
echo "=========================================="
echo "  Sweep Session Complete"
echo "=========================================="
echo "Final sync to Google Drive..."
rsync -a --delete "$LOCAL_RUNS/" "$DRIVE_DIR/runs/"
echo ""
echo "Summary:"
echo "  ✓ Completed: $COMPLETED"
echo "  ✗ Failed: $FAILED"
echo "  Total: $((COMPLETED + FAILED))"
echo "=========================================="

## Check Progress

Run this anytime to see current status:

In [ ]:
# Check progress across all runs in Google Drive
import json
import os
from pathlib import Path
from collections import Counter

DRIVE_RUNS = "/content/drive/MyDrive/lora_scan_results/runs"

if not os.path.exists(DRIVE_RUNS):
    print("No runs found in Google Drive yet.")
else:
    statuses = []
    failed_runs = []
    running_runs = []
    
    for run_dir in Path(DRIVE_RUNS).iterdir():
        if run_dir.is_dir():
            metadata_file = run_dir / "metadata.json"
            if metadata_file.exists():
                with open(metadata_file) as f:
                    meta = json.load(f)
                    status = meta.get("status", "unknown")
                    statuses.append(status)
                    
                    if status == "failed":
                        failed_runs.append({
                            "run_id": meta.get("run_id"),
                            "experiment": meta.get("experiment_id"),
                            "error": meta.get("error", "Unknown error")
                        })
                    elif status == "running":
                        running_runs.append({
                            "run_id": meta.get("run_id"),
                            "experiment": meta.get("experiment_id")
                        })
    
    counts = Counter(statuses)
    total = 72
    completed = counts.get("complete", 0)
    failed = counts.get("failed", 0)
    running = counts.get("running", 0)
    remaining = total - (completed + failed + running)
    
    print("=" * 60)
    print(f"  SWEEP PROGRESS: {completed + failed + running}/{total}")
    print("=" * 60)
    print(f"  ✓ Completed:  {completed}")
    print(f"  ✗ Failed:     {failed}")
    print(f"  ⋯ Running:    {running} (likely stuck from disconnect)")
    print(f"  ○ Remaining:  {remaining}")
    print("=" * 60)
    
    if failed_runs:
        print(f"\n❌ FAILED RUNS ({len(failed_runs)}):")
        for run in failed_runs:
            print(f"  • {run['experiment']}")
            print(f"    ID: {run['run_id']}")
            print(f"    Error: {run['error'][:80]}...")
            print()
    
    if running_runs:
        print(f"\n⚠️  STUCK RUNS ({len(running_runs)}) - likely from disconnection:")
        for run in running_runs:
            print(f"  • {run['experiment']} (ID: {run['run_id']})")
        print("\n💡 Tip: Delete these from Drive to re-run them (see cleanup cell below)")
    
    print(f"\n✨ Progress: {((completed / total) * 100):.1f}% complete")

## Clean Up Failed/Stuck Runs

Use this to delete failed or stuck runs so they can be re-run:

In [ ]:
# Clean up failed and stuck runs from Google Drive
import json
import shutil
from pathlib import Path

DRIVE_RUNS = "/content/drive/MyDrive/lora_scan_results/runs"

# Set what to clean up
CLEAN_FAILED = True   # Delete failed runs
CLEAN_STUCK = True    # Delete stuck "running" runs (from disconnects)

if not os.path.exists(DRIVE_RUNS):
    print("No runs directory found.")
else:
    deleted = []
    
    for run_dir in Path(DRIVE_RUNS).iterdir():
        if run_dir.is_dir():
            metadata_file = run_dir / "metadata.json"
            if metadata_file.exists():
                with open(metadata_file) as f:
                    meta = json.load(f)
                    status = meta.get("status")
                    
                    should_delete = (
                        (status == "failed" and CLEAN_FAILED) or
                        (status == "running" and CLEAN_STUCK)
                    )
                    
                    if should_delete:
                        print(f"Deleting {status} run: {meta.get('experiment_id')} (ID: {meta.get('run_id')})")
                        shutil.rmtree(run_dir)
                        deleted.append(meta.get('experiment_id'))
    
    if deleted:
        print(f"\n✓ Deleted {len(deleted)} run(s)")
        print(f"These will be re-run next time you start the sweep.")
    else:
        print("No runs to clean up.")

## Alternative: Run in batches

If you want more control, you can run the sweep in smaller batches:

In [ ]:
# Run only specific model sizes (easier to fit in Colab time limits)
# Example: Run only qwen25-0.5b experiments
!./bin/micromamba run -n spar python scripts/run.py configs/experiments/sft_sweep_mmlu_gsm8k.yaml --only "qwen25-0.5b"

# Sync after this batch
!rsync -a --delete /content/lora_scan/runs/ /content/drive/MyDrive/lora_scan_results/runs/
print("✓ Batch complete and saved to Drive")

## Download Results Archive

After completion (or at any point), download a snapshot:

In [ ]:
# Create and download results archive
!tar -czf results_$(date +%Y%m%d_%H%M%S).tar.gz runs/

from google.colab import files
import glob

# Download the most recent results file
result_files = sorted(glob.glob('results_*.tar.gz'))
if result_files:
    latest = result_files[-1]
    print(f"Downloading {latest}...")
    files.download(latest)
else:
    print("No results files found")

## Keep-Alive Script (Optional)

Run this to prevent Colab from disconnecting due to inactivity:

In [ ]:
%%javascript
// Keep the session alive by clicking the connect button periodically
function ClickConnect(){
    console.log("Keeping the session alive...");
    document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)  // Click every 60 seconds

## Analyze Results

After the sweep completes (or partially completes):

In [ ]:
# Load and display results
import sys
sys.path.insert(0, 'src')

from lorascan import runs
import pandas as pd

results = runs.collect_results()
df = pd.DataFrame(results)
print(f"\nCompleted runs: {len(df)}/72")
print(df.head())

# Summary statistics
print("\n=== Results Summary ===")
if len(df) > 0:
    summary = df.groupby(['model', 'dataset', 'prompt'])['accuracy'].agg(['mean', 'std', 'count'])
    print(summary)
else:
    print("No completed runs yet")